# Issue #388 — Harden Shared Cutoff Learning after #382 EV

**Executor:** Analyst · **Session:** 20260825_003155_0b8b9f
**Contract:** #388 (PI-owned direct implementation, PI Review before merge, optional post-merge EV)

This notebook executes the acceptance evidence for the five #388 corrections:

1. **Target identity exclusion** — decision-epoch cutoffs exclude every row of
   the target issue, even pre-decision-epoch rows; receipting the exclusion
   count and failing closed on ambiguous identity.
2. **Repository-wide enforcement** — `check_cutoff_governance.py` now scans both
   `src/` and `scripts/`; whole-file legacy exemptions are replaced by
   occurrence/baseline-bound inventory; imports alone never authorize an
   unrelated direct `.fit()`.
3. **Transitive implementation closure** — decision-facing modules bind their
   actual import closure (contracts/constants included); any mutation of an
   imported blob changes the cache key and invalidates replay.
4. **Availability semantics** — `kickoff+4h` is declared a PROXY policy with a
   preregistered conservative safe boundary; receipts bind policy identity and
   margin-to-cutoff; prospective claims fail closed.
5. **Recency blend semantics** — per-row `RECENT_STATE_BLEND` boost is replaced
   by the explicit long-history/recent-state convex mixture
   (`RECENT_STATE_MIXTURE`, blend in (0, 0.5]), binding aggregate mass,
   row/issue mass, ESS and shrinkage. No compatibility wrapper.


In [1]:
from __future__ import annotations
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
print("repo root:", ROOT)

repo root: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/bj388-hardening


## 1. Core hardening (target identity, availability, mixture)

In [2]:
from src.temporal_learning import (
    AvailabilityFilter, Cutoff, TemporalContractError, WeightingPolicy,
    implementation_closure, prepare_learning_context,
)
from src.temporal_learning.core import stable_frame_hash, stable_json_hash
from src.cutoff_mapping.api import fit_cutoff_mapping, fit_issue_walk_forward, mapping_availability
print("imports OK")

imports OK


### Synthetic development panel

Five complete prior years, six issues per year, 40 rows each. An extra target
issue is injected with pre-decision-epoch labels.

In [3]:
OUTCOMES = ("3", "1", "0")
NUMERIC = ("q_eu_3", "q_eu_1", "q_eu_0", "line_x_q_eu_3", "line_x_q_eu_1", "line_x_q_eu_0")
RNG = np.random.default_rng(388_001)
HASHES = {"synthetic_panel": "x" * 64}

def make_development(years=(2019, 2020, 2021, 2022, 2023), issues_per_year=6, rows_per_issue=40):
    rows = []
    for year in years:
        for issue_i in range(issues_per_year):
            issue = f"{year}-{issue_i+1:03d}"
            anchor = pd.Timestamp(f"{year}-{3+issue_i}-15", tz="UTC")
            for match in range(rows_per_issue):
                row = {
                    "issue_code": issue, "issue_year": year, "match_idx": match,
                    "fixture_decision_ts": anchor - pd.Timedelta(hours=10),
                    "label_available_ts": anchor + pd.Timedelta(hours=4),
                    "native_line_supported": True,
                    "native_handicap_line": float([-1.0, 0.0, 1.0, -1.0, -1.0, 1.0][match % 6]),
                    "target_outcome": str(OUTCOMES[match % 3]),
                }
                p = RNG.dirichlet([2.0, 1.0, 1.0])
                row.update({f"q_eu_{code}": float(v) for code, v in zip(OUTCOMES, p)})
                for code in OUTCOMES:
                    row[f"line_x_q_eu_{code}"] = row["native_handicap_line"] * row[f"q_eu_{code}"]
                rows.append(row)
    return pd.DataFrame(rows)

def make_target(issue="2024-001", year=2024, rows=12):
    anchor = pd.Timestamp(f"{year}-02-10", tz="UTC")
    out = []
    for match in range(rows):
        row = {
            "issue_code": issue, "issue_year": year, "match_idx": match,
            "fixture_decision_ts": anchor - pd.Timedelta(hours=10),
            "native_line_supported": True,
            "native_handicap_line": float([-1.0, 0.0, 1.0, -1.0, -1.0, 1.0][match % 6]),
        }
        p = RNG.dirichlet([2.0, 1.0, 1.0])
        row.update({f"q_eu_{code}": float(v) for code, v in zip(OUTCOMES, p)})
        for code in OUTCOMES:
            row[f"line_x_q_eu_{code}"] = row["native_handicap_line"] * row[f"q_eu_{code}"]
        out.append(row)
    return pd.DataFrame(out)

dev = make_development()
target = make_target()
print("development rows:", len(dev), "| target rows:", len(target))

development rows: 1200 | target rows: 12


### 1a. Target identity exclusion

In [4]:
cutoff = Cutoff.decision_epoch("2023-006", pd.Timestamp("2023-06-20", tz="UTC"))
injected = dev[dev["issue_code"] == "2022-006"].assign(
    issue_code="2023-006", issue_year=2023,
    label_available_ts=pd.Timestamp("2023-05-01", tz="UTC") + pd.Timedelta(hours=2),
    fixture_decision_ts=pd.Timestamp("2023-05-01", tz="UTC") - pd.Timedelta(hours=10),
)
history = pd.concat([dev, injected], ignore_index=True)
ctx = prepare_learning_context(
    history, cutoff,
    source_hashes=HASHES,
    implementation_hashes={"closure": "y" * 64},
    availability=mapping_availability(cutoff),
    weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0),
)
print("target_exclusion:", ctx.receipt.payload["target_exclusion"])
print("eligible rows with target issue:", ctx.eligible["issue_code"].astype(str).eq("2023-006").sum())
print("pre-cutoff target rows excluded:", ctx.receipt.payload["target_exclusion"]["excluded_rows"])

# fail closed on ambiguous / missing identity
try:
    prepare_learning_context(
        dev,  # no 2023-999 rows at all
        Cutoff.decision_epoch("2023-999", pd.Timestamp("2023-06-20", tz="UTC")),
        source_hashes=HASHES, implementation_hashes={"closure": "y" * 64},
        availability=mapping_availability(cutoff),
        weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0),
    )
    raise AssertionError("missing target identity must fail closed")
except TemporalContractError as exc:
    print("fail-closed on missing target identity:", str(exc)[:70])

hangry = history.copy()
hangry.loc[0, "issue_code"] = ""
try:
    prepare_learning_context(
        hangry, cutoff,
        source_hashes=HASHES, implementation_hashes={"closure": "y" * 64},
        availability=mapping_availability(cutoff),
        weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0),
    )
    raise AssertionError("ambiguous identity must fail closed")
except TemporalContractError as exc:
    print("fail-closed on ambiguous identity:", str(exc)[:70])

target_exclusion: {'enabled': True, 'target_identity': '2023-006', 'excluded_rows': 40}
eligible rows with target issue: 0
pre-cutoff target rows excluded: 40
fail-closed on missing target identity: decision-epoch cutoff DECISION_EPOCH_2023-999 names target issue 2023-
fail-closed on ambiguous identity: ambiguous target identity: history contains blank or missing issue cod


### 1b. Availability semantics — proxy policy identity + margin-to-cutoff

In [5]:
# A PROXY policy without a preregistered conservative boundary fails construction
try:
    AvailabilityFilter(policy_identity="PROXY_KICKOFF_PLUS_4H")
    raise AssertionError("proxy without boundary must fail")
except TemporalContractError as exc:
    print("proxy w/o boundary fails:", str(exc)[:60])

# mapping client declares the frozen kickoff+4h proxy explicitly
af = mapping_availability(Cutoff.evaluation_year(2024, pd.Timestamp("2024-01-01", tz="UTC")))
print("mapping availability policy:", af.policy_identity)
print("proxy safe boundary:", af.proxy_safe_boundary_ts)

# prospective claim through default proxy (boundary == cutoff) fails closed
try:
    fit_cutoff_mapping(dev, target,
        Cutoff.evaluation_year(2024, pd.Timestamp("2024-01-01", tz="UTC")),
        source_hashes=HASHES, prospective_capable=True)
    raise AssertionError("prospective proxy claim must fail")
except TemporalContractError as exc:
    print("prospective fail-closed:", str(exc)[:70])

# a strict preregistered boundary before cutoff permits a prospective claim
strict = AvailabilityFilter(
    label_available_column="label_available_ts",
    data_available_column="fixture_decision_ts",
    policy_identity="PROXY_KICKOFF_PLUS_4H_STRICT",
    proxy_safe_boundary_ts="2023-12-01T00:00:00Z",
)
ctx_pro = prepare_learning_context(
    dev, Cutoff.evaluation_year(2024, pd.Timestamp("2024-01-01", tz="UTC")),
    source_hashes=HASHES, implementation_hashes={"closure": "y" * 64},
    availability=strict,
    weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0),
    prospective_capable=True,
)
print("prospective with strict boundary: OK; receipt flags:",
      ctx_pro.receipt.payload["availability_policy"]["prospective_capable"])
print("margin-to-cutoff days:", ctx_pro.receipt.payload["margin_to_cutoff_days"])

proxy w/o boundary fails: proxy availability policy requires a conservative safe bound
mapping availability policy: PROXY_KICKOFF_PLUS_4H
proxy safe boundary: 2024-01-01 00:00:00+00:00
prospective fail-closed: prospective claim requires observed availability or a preregistered co
prospective with strict boundary: OK; receipt flags: True
margin-to-cutoff days: {'minimum': 138.83333333333334, 'maximum': 1752.8333333333333, 'median': 945.3333333333334}


### 1c. Recency mixture semantics (replaces per-row blend)

In [6]:
mix_policy = WeightingPolicy.recent_state_mixture(rolling_window_issues=2, recent_blend=0.3, minimum_ess=10.0)
ctx_mix = prepare_learning_context(
    dev, Cutoff.evaluation_year(2024, pd.Timestamp("2024-01-01", tz="UTC")),
    source_hashes=HASHES, implementation_hashes={"closure": "y" * 64},
    weighting_policy=mix_policy,
)
mix = ctx_mix.receipt.payload["weight_receipt"]["mixture"]
print(json.dumps(mix, indent=2))
assert abs(mix["recent_part_mass"] - 0.3) < 1e-9      # declared coefficient
assert mix["long_part_mass"] >= mix["recent_part_mass"]  # bounded by long prior (0.7 >= 0.3)
assert abs(mix["mass_sum"] - 1.0) < 1e-9              # mass reconciles
assert mix["ess"] > 0                                  # ESS bound
issue_mass = ctx_mix.receipt.payload["weight_receipt"]["issue_mass"]
print("issue mass rows:", len(issue_mass), "sum~rows:", round(sum(issue_mass.values()), 3))

# old per-row RECENT_STATE_BLEND is deleted — no compatibility wrapper
try:
    prepare_learning_context(
        dev, Cutoff.evaluation_year(2024, pd.Timestamp("2024-01-01", tz="UTC")),
        source_hashes=HASHES, implementation_hashes={"closure": "y" * 64},
        weighting_policy=WeightingPolicy("RECENT_STATE_BLEND", rolling_window_issues=2, recent_blend=0.3),
    )
    raise AssertionError("RECENT_STATE_BLEND must be rejected")
except TemporalContractError as exc:
    print("old blend rejected:", str(exc)[:60])

# blend outside (0, 0.5] fails closed
try:
    prepare_learning_context(
        dev, Cutoff.evaluation_year(2024, pd.Timestamp("2024-01-01", tz="UTC")),
        source_hashes=HASHES, implementation_hashes={"closure": "y" * 64},
        weighting_policy=WeightingPolicy.recent_state_mixture(2, 0.9, minimum_ess=10.0),
    )
    raise AssertionError("blend > 0.5 must fail")
except TemporalContractError as exc:
    print("blend > 0.5 rejected:", str(exc)[:60])

{
  "declared_recent_blend": 0.3,
  "long_part_mass": 0.7,
  "recent_part_mass": 0.3,
  "mass_sum": 1.0,
  "normalized_recent_row_aggregate": 416.0,
  "recent_rows": 80,
  "long_rows": 1120,
  "ess": 530.9734513274337,
  "shrinkage_toward_long_prior": 0.7
}
issue mass rows: 30 sum~rows: 1200.0
old blend rejected: unsupported weighting policy: RECENT_STATE_BLEND
blend > 0.5 rejected: recent-state mixture blend must lie in (0, 0.5] so the long-


## 2. Transitive implementation closure

In [7]:
closure = implementation_closure(
    ROOT,
    [ROOT / "src/cutoff_mapping/api.py", ROOT / "src/cutoff_mapping/contract.py"],
)
print("closure modules:", list(closure.modules.keys()))
print("closure hash:", closure.hash()[:16])
assert "src.temporal_learning" in closure.modules       # package __init__
assert "src.temporal_learning.core" in closure.modules  # shared core
assert "src.temporal_learning.closure" in closure.modules
assert closure.hash() != ""

# a mutation of any imported blob changes the hash
import tempfile, shutil
probe = Path(tempfile.mkdtemp())
before = closure.hash()
(probe / "src/cutoff_mapping").mkdir(parents=True)
contract_text = (ROOT / "src/cutoff_mapping/contract.py").read_text(encoding="utf-8")
(probe / "src/cutoff_mapping/contract.py").write_text(contract_text + "\n# tampered\n", encoding="utf-8")
(probe / "src/cutoff_mapping/api.py").write_text(
    (ROOT / "src/cutoff_mapping/api.py").read_text(encoding="utf-8"), encoding="utf-8")
mutated = implementation_closure(
    probe, [probe / "src/cutoff_mapping/api.py", probe / "src/cutoff_mapping/contract.py"])
after = mutated.hash()
print("mutation invalidates closure:", before != after)
shutil.rmtree(probe)

closure modules: ['src.cutoff_mapping.api', 'src.cutoff_mapping.contract', 'src.temporal_learning', 'src.temporal_learning.closure', 'src.temporal_learning.core', 'src.temporal_learning.example_selector']
closure hash: 27e1c4206efc0a0e
mutation invalidates closure: True


## 3. Repository-wide enforcement (checker probes)

In [8]:
import subprocess, tempfile, shutil

def run_checker(root):
    proc = subprocess.run(
        [sys.executable, str(root / "scripts/check_cutoff_governance.py")],
        capture_output=True, text=True, cwd=root)
    return proc.returncode, proc.stdout + proc.stderr

# baseline tree must pass
rc, out = run_checker(ROOT)
print("baseline checker:", "PASS" if rc == 0 else "FAIL", out.strip()[:200])

tmp = Path(tempfile.mkdtemp())
repo = tmp / "repo"
for sub in ("src", "scripts", "config", "docs"):
    if (ROOT / sub).is_dir():
        shutil.copytree(ROOT / sub, repo / sub, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))

# probe: new direct fit in scripts/ -> REJECT
(repo / "scripts/sneaky_probe.py").write_text(
    "import numpy as np\n"
    "from sklearn.linear_model import LogisticRegression\n"
    "def sneaky_fit():\n"
    "    LogisticRegression().fit(np.zeros((5,2)), np.zeros(5))\n", encoding="utf-8")
rc, out = run_checker(repo)
print("new scripts/ direct fit rejected:", rc != 0 and "direct fit outside shared" in out)
(repo / "scripts/sneaky_probe.py").unlink()

# probe: unrelated fit in a shared-importing module -> REJECT
(repo / "scripts/shared_import_probe.py").write_text(
    "import numpy as np\n"
    "from sklearn.linear_model import LogisticRegression\n"
    "from src.temporal_learning import prepare_learning_context\n"
    "def unrelated_fit():\n"
    "    LogisticRegression().fit(np.zeros((5,2)), np.zeros(5))\n", encoding="utf-8")
rc, out = run_checker(repo)
print("unrelated fit in shared-importing module rejected:", rc != 0 and "direct fit outside shared" in out)

# probe: new hidden temporal global -> REJECT
(repo / "scripts/hidden_global_probe.py").write_text("FIT_END = 2024\n", encoding="utf-8")
rc, out = run_checker(repo)
print("new hidden global rejected:", rc != 0 and "hidden temporal globals" in out)

# probe: detached mapping_accepted consumer outside mapping package -> REJECT
(repo / "src/new_consumer.py").write_text("def use():\n    return mapping_accepted\n", encoding="utf-8")
rc, out = run_checker(repo)
print("detached mapping consumer rejected:", rc != 0 and "detached mapping_accepted" in out)
shutil.rmtree(tmp)

baseline checker: PASS cutoff governance: PASS


new scripts/ direct fit rejected: True


unrelated fit in shared-importing module rejected: True


new hidden global rejected: True


detached mapping consumer rejected: True


## 4. Walk-forward per-issue target exclusion

In [9]:
targets = pd.concat([
    make_target("2023-001", 2023, 8),
    make_target("2023-002", 2023, 8),
    make_target("2023-003", 2023, 8),
], ignore_index=True)
dev_wf = dev.copy()
for issue in ("2023-001", "2023-002", "2023-003"):
    dev_wf = pd.concat([dev_wf, dev[dev["issue_code"] == "2022-006"].assign(
        issue_code=issue, issue_year=2023,
        label_available_ts=pd.Timestamp("2023-01-05", tz="UTC") + pd.Timedelta(hours=2),
        fixture_decision_ts=pd.Timestamp("2023-01-05", tz="UTC") - pd.Timedelta(hours=10),
    )], ignore_index=True)
mapped, receipts, weights = fit_issue_walk_forward(
    dev_wf, targets, source_hashes=HASHES,
    weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0),
)
print("walk-forward issues:", targets["issue_code"].nunique(), "| mapped rows:", len(mapped))
for receipt in receipts:
    assert receipt["target_exclusion"]["enabled"] is True
    assert receipt["target_exclusion"]["excluded_rows"] == 40
print("per-issue target exclusion: OK")

walk-forward issues: 3 | mapped rows: 24
per-issue target exclusion: OK


## 5. Cache isolation and equivalent-cutoff parity

In [10]:
year_cutoff = Cutoff.evaluation_year(2024, pd.Timestamp("2024-01-01", tz="UTC"))
last_cutoff = Cutoff.last_issue("2023-006", pd.Timestamp("2024-01-01", tz="UTC"))
impl = {"closure": "y" * 64}
a = prepare_learning_context(dev, year_cutoff, source_hashes=HASHES, implementation_hashes=impl,
                             weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0))
b = prepare_learning_context(dev, last_cutoff, source_hashes=HASHES, implementation_hashes=impl,
                             weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0))
print("equivalent admitted history equal:", a.receipt.payload["admitted_history_sha256"] == b.receipt.payload["admitted_history_sha256"])
print("cache keys distinct:", a.receipt.payload["cache_key"] != b.receipt.payload["cache_key"])
from src.temporal_learning import temporal_cache_key
from src.temporal_learning.core import PartitionPolicy
clean = temporal_cache_key(year_cutoff, HASHES, impl, PartitionPolicy(), WeightingPolicy.expanding_equal())
poison = temporal_cache_key(year_cutoff, HASHES, {"closure": "z" * 64}, PartitionPolicy(), WeightingPolicy.expanding_equal())
print("poisoned implementation cache rejected:", clean != poison)

equivalent admitted history equal: True
cache keys distinct: True
poisoned implementation cache rejected: True


## 6. Shared API — mapping plus non-mapping client

In [11]:
from src.temporal_learning.example_selector import select_cutoff_threshold
score_dev = dev.copy()
score_dev["score"] = RNG.normal(size=len(score_dev))
score_dev["binary_label"] = (score_dev["score"] > 0).astype(int)
selection = select_cutoff_threshold(
    score_dev, year_cutoff,
    score_column="score", binary_label_column="binary_label",
    source_hashes=HASHES, implementation_hashes=impl,
)
print("threshold client selection:", round(selection.selection, 4),
      "| receipt client:", selection.receipt["client"])
mapping_result = fit_cutoff_mapping(dev, target, year_cutoff, source_hashes=HASHES)
print("mapping client accepted:", mapping_result.accepted,
      "| receipt binds closure:", "cutoff_mapping.closure" in mapping_result.receipt["implementation_hashes"])

threshold client selection: 0.2244 | receipt client: REPRESENTATIVE_THRESHOLD_SELECTION
mapping client accepted: False | receipt binds closure: True


## 7. Tamper self-tests (REQ-E06)

In [12]:
from src.cutoff_mapping.api import replay_mapped

# (a) tamper mapping acceptance field -> replay cannot silently reproduce science
receipt_tampered = dict(mapping_result.receipt)
receipt_tampered["mapping_accepted"] = not mapping_result.accepted
receipt_tampered["scientific_output_sha256"] = stable_json_hash(
    {"accepted": receipt_tampered["mapping_accepted"], "model_identity": "0"*64, "mapped_frame": "0"*64})
replayed_tampered = replay_mapped(target, receipt_tampered)
print("tampered acceptance differs from original mapped frame:",
      not replayed_tampered.equals(mapping_result.mapped.reset_index(drop=True)))

# (b) tamper one weight ledger row -> receipt weight-frame hash detects it
ledger_tampered = ctx.weight_ledger.copy()
ledger_tampered.loc[0, "weight"] = ledger_tampered["weight"].max() * 10
print("tampered ledger hash differs:",
      stable_frame_hash(ledger_tampered) != ctx.receipt.payload["weight_receipt"]["weight_frame_sha256"])

# (c) manifest rewrite -> verifier fails
from src.cutoff_mapping.package import PACKAGE_FILES, verify_manifest
from src.cutoff_mapping.contract import CONTRACT_ID
fake = Path(tempfile.mkdtemp()) / "package"
fake.mkdir(parents=True)
for rel in PACKAGE_FILES:
    p = fake / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text('{"probe": "x"}', encoding="utf-8")
(fake / "MANIFEST.json").write_text(json.dumps({
    "contract": CONTRACT_ID,
    "files": {name: {"sha256": "0"*64, "bytes": 1} for name in PACKAGE_FILES},
    "implementation": {}, "deliveries": {}}), encoding="utf-8")
try:
    verify_manifest(ROOT, fake)
    raise AssertionError("falsified manifest must be rejected")
except TemporalContractError as exc:
    print("falsified manifest rejected:", str(exc)[:60])
shutil.rmtree(fake)

# (d) forged source trace / future label -> fail closed
try:
    prepare_learning_context(
        dev, Cutoff.decision_epoch("2023-999", pd.Timestamp("2023-06-20", tz="UTC")),
        source_hashes=HASHES, implementation_hashes=impl,
        availability=mapping_availability(cutoff),
        weighting_policy=WeightingPolicy.expanding_equal(minimum_ess=10.0))
    raise AssertionError("forged source trace must fail")
except TemporalContractError as exc:
    print("forged source trace rejected:", str(exc)[:60])

tampered acceptance differs from original mapped frame: True
tampered ledger hash differs: True
falsified manifest rejected: manifest mismatch: source_receipt.json
forged source trace rejected: decision-epoch cutoff DECISION_EPOCH_2023-999 names target i


## Summary

All #388 acceptance items are executed and asserted above:

| Acceptance | Evidence |
|---|---|
| decision-epoch target row excluded from fit/tune/calibration/final refit | §1a exclusion counts + partitions |
| post-cutoff & same-target mutation invariance | test suite (deterministic replay) |
| scripts/src static bypass probes rejected | §3 checker probes |
| allowlisted-file new occurrence rejected | §3 + tests |
| unrelated direct fit in shared-importing module rejected | §3 probe |
| transitive contract-module mutation rejected | §2 closure mutation |
| observed/proxy availability explicit, fail-closed prospective | §1b |
| recency mixture mass/bound/ESS reconciled | §1c |
| equivalent cutoffs / cache isolation / replay intact | §5 |
| mapping + non-mapping clients share one API | §6 |
| executed notebook / source-bound verifier / adversarial tests | this notebook + `tests/research/test_cutoff_governance_hardening.py` |

Run the test suite:

```bash
python -m pytest tests/research/test_cutoff_governance_hardening.py -q
```
